# DriveSense-VLM — 05: Gradio Demo

**GPU**: A100 or T4 | **Time**: ~5 min | **Cost**: ~2 CU

Interactive Gradio demo: upload a dashcam image, get structured hazard detection JSON with bounding-box visualization.

> ⚠️ **Before running**: Runtime → Change runtime type → **T4 GPU** (or A100)
>
> **Prerequisites**: `02_optimization.ipynb` must have produced the quantized model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)

!ln -sfn {PROJECT_ROOT}/data {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs
sys.path.insert(0, f"{REPO_ROOT}/src")

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ Repo root    : {REPO_ROOT}")
print(f"✓ Outputs root : {OUTPUTS_ROOT}")

In [ ]:
!pip install gradio transformers peft accelerate "bitsandbytes>=0.46.1" "torchao>=0.16.0" Pillow -q 2>&1 | tail -3

import torch
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

assert os.path.exists(f"{OUTPUTS_ROOT}/quantized_model"), \
    "Missing quantized_model — run notebook 02_optimization.ipynb first"
print("✓ Quantized model found")

import drivesense
print(f"✓ drivesense package imported")

In [ ]:
import json, glob, re, time
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image, ImageDraw
import gradio as gr

# ── Config ────────────────────────────────────────────────────────────────────
QUANTIZED_MODEL_DIR = f"{OUTPUTS_ROOT}/quantized_model"
MERGED_MODEL_DIR    = f"{OUTPUTS_ROOT}/merged_model"
EXAMPLE_IMAGES = sorted(glob.glob(f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images/*.jpg"))[:6]

PROMPT = (
    "Analyze this dashcam image for safety hazards. Return JSON with hazards array "
    "containing bbox_2d (normalized 0-1000), label, severity (low/medium/high/critical), "
    "reasoning, and action for each hazard. Include scene_summary and ego_context "
    "(weather, time_of_day, road_type)."
)

SEVERITY_COLORS = {
    "critical": (255,   0,   0),
    "high":     (255, 140,   0),
    "medium":   (255, 215,   0),
    "low":      ( 50, 205,  50),
}

# ── Load model (once) ─────────────────────────────────────────────────────────
print("Loading NF4 quantized model…")
_processor = AutoProcessor.from_pretrained(MERGED_MODEL_DIR)
_model = AutoModelForImageTextToText.from_pretrained(
    QUANTIZED_MODEL_DIR, device_map="auto", torch_dtype=torch.bfloat16,
)
_model.eval()
print(f"✓ Model loaded  |  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Helpers ───────────────────────────────────────────────────────────────────
def _parse_json(text: str) -> dict:
    """Extract first JSON object; strip ```json fences."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1] if "\n" in text else text
        if text.endswith("```"):
            text = text[:-3].rstrip()
    start, end = text.find("{"), text.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return {"hazards": [], "scene_summary": text, "ego_context": {}}


def _draw_boxes(image: Image.Image, ann: dict) -> Image.Image:
    """Overlay severity-coded bounding boxes on a PIL image."""
    w, h = image.size
    base    = image.convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    draw    = ImageDraw.Draw(overlay)
    for hazard in ann.get("hazards", []):
        bbox = hazard.get("bbox_2d", [])
        if len(bbox) != 4:
            continue
        sev   = str(hazard.get("severity", "low")).lower()
        label = str(hazard.get("label", "hazard"))
        color = SEVERITY_COLORS.get(sev, (65, 105, 225))
        x1 = int(bbox[0] * w / 1000)
        y1 = int(bbox[1] * h / 1000)
        x2 = int(bbox[2] * w / 1000)
        y2 = int(bbox[3] * h / 1000)
        draw.rectangle([x1, y1, x2, y2], fill=(*color, 50))
        draw.rectangle([x1, y1, x2, y2], outline=(*color, 255), width=2)
        draw.text((x1 + 2, max(0, y1 - 18)), f"{label} ({sev})", fill=(*color, 255))
    return Image.alpha_composite(base, overlay).convert("RGB")


def analyze(image, max_tokens: int):
    if image is None:
        return None, "Upload a dashcam image.", "—"
    image = image.convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text":  PROMPT},
    ]}]
    text   = _processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _processor(text=[text], images=[image], return_tensors="pt").to("cuda")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    torch.cuda.synchronize()
    ms  = (time.perf_counter() - t0) * 1000
    raw = _processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ann = _parse_json(raw)
    annotated = _draw_boxes(image, ann)
    n = len(ann.get("hazards", []))
    return annotated, json.dumps(ann, indent=2), f"✓ {n} hazard(s) detected  |  {ms:.0f} ms"


# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title="DriveSense-VLM", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DriveSense-VLM: Autonomous Vehicle Hazard Detection")
    gr.Markdown(
        "SFT-optimized Qwen2.5-VL-3B for rare hazard detection. NF4 4-bit quantized (2.4 GB)."
    )

    with gr.Row():
        with gr.Column():
            img_in     = gr.Image(label="Dashcam Frame", type="pil")
            max_tok    = gr.Slider(50, 500, value=300, step=10, label="Max tokens")
            run_btn    = gr.Button("Detect Hazards", variant="primary")
            status_lbl = gr.Textbox(label="Status", interactive=False)
        with gr.Column():
            img_out  = gr.Image(label="Annotated Detection", type="pil")
            json_out = gr.Code(
                label="Structured JSON Output", language="json", lines=20
            )

    run_btn.click(
        fn=analyze,
        inputs=[img_in, max_tok],
        outputs=[img_out, json_out, status_lbl],
    )

    if EXAMPLE_IMAGES:
        gr.Examples(
            examples=[[p] for p in EXAMPLE_IMAGES],
            inputs=[img_in],
            label="Example Dashcam Frames",
        )

demo.launch(share=True, debug=False)

In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("✓ HF_TOKEN set")

In [ ]:
os.chdir(REPO_ROOT)
!git pull

# Copy example images first
import shutil, glob, os
os.makedirs(f"{REPO_ROOT}/demo/examples", exist_ok=True)
images = sorted(glob.glob(f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images/*.jpg"))[:6]
for img in images:
    shutil.copy(img, f"{REPO_ROOT}/demo/examples/")
print(f"✅ Copied {len(images)} example images")

# Upload to HuggingFace
!pip install huggingface_hub -q
from huggingface_hub import login
login()  # will prompt for your HF token

!python scripts/upload_to_hf.py \
    --model-dir {OUTPUTS_ROOT}/quantized_model \
    --processor-dir {OUTPUTS_ROOT}/merged_model \
    --examples-dir {REPO_ROOT}/demo/examples \
    --repo-id jayanth7111/DriveSense-VLM

In [ ]:
import glob, os
from IPython.display import Image as IPImage, display

# Get test images sorted
images = sorted(glob.glob(f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images/*.jpg"))
print(f"Total available: {len(images)} images\n")

# Display first 10 to visually pick a good one
for i, img_path in enumerate(images[:10]):
    print(f"\n[{i}] {os.path.basename(img_path)}")
    display(IPImage(filename=img_path, width=400))

In [ ]:
# Pick the index you liked
INDEX = 2  # change this
selected = images[INDEX]
print(f"Use this in Gradio: {selected}")

# Or download it to upload via the public Gradio link
from google.colab import files
files.download(selected)